# Multi-Crop Monitoring System — Phase 3A Disease Classifiers
## Google Colab T4 GPU Training Notebook

This notebook trains **EfficientNet-B0** disease classifiers for:
1. **Sugarcane (SPPU)** — 5 classes (`Healthy`, `Mosaic`, `RedRot`, `Rust`, `Yellow`), evaluated strictly on deduplicated `use_for_eval == true` rows.
2. **Wheat (Zenodo / JIC)** — 5 classes (`BrownRust`, `Healthy`, `Mildew`, `Septoria`, `YellowRust`).
3. **Cotton (Mendeley)** — 4 classes (`bacterial blight`, `curl virus`, `fussarium wilt`, `healthy`) with **Train-split class weights** and **Stratified 5-Fold Cross-Validation** on train+val (882 images).

### Features:
- **Google Drive Integration**: Mounts Drive, reads zipped datasets, writes checkpoints & reports directly to Drive.
- **Disconnection Resilience**: Every epoch saves `best_checkpoint.pt` and `last_checkpoint.pt`. Re-running with `--resume` resumes instantly from the last completed epoch.
- **Strict No Fake Data Policy**: Real evaluation metrics on held-out test split (confusion matrices, accuracy, macro-F1, Wilson 95% CIs).
- **Smoke Test Support**: `--smoke` runs 1 epoch and max 3 batches to verify the full pipeline end-to-end.

### Step 1: Mount Google Drive & Verify GPU Runtime

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Check GPU Availability (T4 recommended)
!nvidia-smi

### Step 2: Clone or Update Project Repository

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/multi-crop-monitoring-system")
BASE_REPO = "github.com/sakshidaund59/multi-crop-monitoring-system.git"

gh_token = None
try:
    from google.colab import userdata
    gh_token = userdata.get("GH_TOKEN")
except Exception:
    pass

if gh_token:
    clone_url = f"https://{gh_token}@{BASE_REPO}"
    print("[INFO] Authenticating clone using GH_TOKEN secret from Colab Secrets.")
else:
    clone_url = f"https://{BASE_REPO}"
    print("[INFO] GH_TOKEN secret not found or accessible; proceeding unauthenticated (public access).")

if not REPO_DIR.exists():
    cmd = ["git", "clone", clone_url, str(REPO_DIR)]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        err_msg = res.stderr.replace(gh_token, "***") if gh_token else res.stderr
        raise RuntimeError(f"Git clone failed: {err_msg}")
    print("[SUCCESS] Repository cloned successfully.")
else:
    print(f"[INFO] Repository directory already exists at {REPO_DIR}.")

%cd /content/multi-crop-monitoring-system
!git pull origin main || true

### Step 3: Install Required Python Packages

In [ ]:
!pip install -q timm torch torchvision matplotlib seaborn scikit-learn pandas pyyaml pyflakes pytest

### Step 4: Unpack Datasets (One Folder per Dataset)
Expects ZIP files located under `/content/drive/MyDrive/multicrop/datasets/`:
- `sugarcane_sppu.zip` (Contents at top level)
- `wheat_zenodo_jic.zip` (Contents at top level)
- `cotton_mendeley.zip` (Contents at top level)

In [ ]:
%%bash
export DATA_ROOT=/content/data
mkdir -p /content/data
mkdir -p /content/drive/MyDrive/multicrop/checkpoints
mkdir -p /content/drive/MyDrive/multicrop/reports

for name in sugarcane_sppu wheat_zenodo_jic cotton_mendeley; do
  ZIP_PATH="/content/drive/MyDrive/multicrop/datasets/${name}.zip"
  TARGET_DIR="/content/data/${name}"
  if [ -f "$ZIP_PATH" ]; then
    echo "Extracting ${name} to ${TARGET_DIR}..."
    mkdir -p "${TARGET_DIR}" && unzip -q -o "${ZIP_PATH}" -d "${TARGET_DIR}"
    echo "[SUCCESS] ${name} extracted."
  else
    echo "[ERROR] Missing zip file: ${ZIP_PATH}"
  fi
done

### Step 5: Configure Environment Variables

In [ ]:
import os
os.environ["DATA_ROOT"] = "/content/data"
os.environ["PYTHONPATH"] = "/content/multi-crop-monitoring-system"
print(f"DATA_ROOT set to: {os.environ['DATA_ROOT']}")

### Step 6: Pre-Training Dataset Path & Integrity Verification
Verifies that every image in the manifests resolves to an existing file under `/content/data` with 0 missing files and 0 backslashes.

In [ ]:
import os
import sys
from pathlib import Path
from ml_training.scripts.classifier_utils import (
    CROP_MANIFEST_MAP,
    load_manifest_rows,
    resolve_image_path,
)

data_root = Path(os.environ.get("DATA_ROOT", "/content/data"))
crops = ["sugarcane", "wheat", "cotton"]
total_errors = 0

print("=" * 80)
print(f" PRE-TRAINING DATASET INTEGRITY VERIFICATION (DATA_ROOT={data_root})")
print("=" * 80)

for crop in crops:
    manifest_name = CROP_MANIFEST_MAP[crop]
    manifest_path = Path("ml_training/manifests") / manifest_name
    if not manifest_path.exists():
        print(f"[FATAL] Manifest missing: {manifest_path}")
        total_errors += 1
        continue

    print(f"\nChecking crop: {crop.upper()} (Manifest: {manifest_name})")
    for split in ["train", "val", "test"]:
        rows = load_manifest_rows(manifest_path, crop=crop, split=split, filter_eval_only=False)
        found_count = 0
        missing_count = 0
        backslash_count = 0

        for r in rows:
            raw_path = r["path"]
            if "\\" in raw_path:
                backslash_count += 1
            resolved = resolve_image_path(data_root, crop, raw_path)
            if resolved.exists():
                found_count += 1
            else:
                missing_count += 1

        print(f"  [{split.upper():<5}] Total: {len(rows):<5} | Found: {found_count:<5} | Missing: {missing_count:<5} | Backslashes: {backslash_count:<5}")

        if backslash_count > 0:
            print(f"  [ERROR] {crop} {split} contains {backslash_count} paths with backslashes!")
            total_errors += backslash_count

        if missing_count > 0:
            print(f"  [ERROR] {crop} {split} has {missing_count} missing images on disk under {data_root}!")
            total_errors += missing_count

print("\n" + "=" * 80)
if total_errors > 0:
    raise RuntimeError(f"Pre-training verification FAILED with {total_errors} errors! STOPPING before training.")
else:
    print("[SUCCESS] All dataset paths verified on disk. 0 missing files, 0 backslashes. Safe to proceed.")
print("=" * 80)

### Step 7: Pipeline Smoke Test (`--smoke`)
Runs 1 epoch with at most 3 batches per epoch through the full training, checkpointing, and evaluation code paths. Output files are labelled: `SMOKE TEST, not valid metrics`.

In [ ]:
!python ml_training/scripts/train_classifier.py \
    --crop sugarcane \
    --data-root /content/data \
    --out-dir /content/smoke_test/sugarcane \
    --smoke

### Step 8: Train & Evaluate Sugarcane Classifier (EfficientNet-B0)

In [ ]:
# Train Sugarcane Classifier (Resume-enabled)
!python ml_training/scripts/train_classifier.py \
    --crop sugarcane \
    --data-root /content/data \
    --out-dir /content/drive/MyDrive/multicrop/checkpoints/sugarcane \
    --epochs 30 \
    --batch-size 32 \
    --lr 5e-4 \
    --seed 42 \
    --resume

In [ ]:
# Evaluate Sugarcane Classifier on Held-out Test Split
!python ml_training/scripts/evaluate_classifier.py \
    --checkpoint /content/drive/MyDrive/multicrop/checkpoints/sugarcane/best_checkpoint.pt \
    --data-root /content/data \
    --crop sugarcane \
    --out-dir /content/drive/MyDrive/multicrop/reports

### Step 9: Train & Evaluate Wheat Classifier (EfficientNet-B0)

In [ ]:
# Train Wheat Classifier (Resume-enabled)
!python ml_training/scripts/train_classifier.py \
    --crop wheat \
    --data-root /content/data \
    --out-dir /content/drive/MyDrive/multicrop/checkpoints/wheat \
    --epochs 30 \
    --batch-size 32 \
    --lr 5e-4 \
    --seed 42 \
    --resume

In [ ]:
# Evaluate Wheat Classifier on Held-out Test Split
!python ml_training/scripts/evaluate_classifier.py \
    --checkpoint /content/drive/MyDrive/multicrop/checkpoints/wheat/best_checkpoint.pt \
    --data-root /content/data \
    --crop wheat \
    --out-dir /content/drive/MyDrive/multicrop/reports

### Step 10: Train & Evaluate Cotton Classifier (EfficientNet-B0)
Includes **Stratified 5-Fold Cross-Validation** on train+val (882 images) and **Train-split class weights**.

In [ ]:
# Train Cotton Classifier (Runs 5-fold CV + Full Model Training with Resume)
!python ml_training/scripts/train_classifier.py \
    --crop cotton \
    --data-root /content/data \
    --out-dir /content/drive/MyDrive/multicrop/checkpoints/cotton \
    --epochs 30 \
    --batch-size 32 \
    --lr 5e-4 \
    --seed 42 \
    --cv \
    --resume

In [ ]:
# Evaluate Cotton Classifier on Held-out Test Split
# Note: Output explicitly labelled 'indicative only (tiny classes: curl virus test n=2, fusarium wilt test n=5)'
!python ml_training/scripts/evaluate_classifier.py \
    --checkpoint /content/drive/MyDrive/multicrop/checkpoints/cotton/best_checkpoint.pt \
    --data-root /content/data \
    --crop cotton \
    --out-dir /content/drive/MyDrive/multicrop/reports

### Step 11: Verify Saved Artifacts on Google Drive

In [ ]:
print("=== SAVED CHECKPOINTS ON GOOGLE DRIVE ===")
!ls -lh /content/drive/MyDrive/multicrop/checkpoints/*/*.pt

print("\n=== SAVED EVALUATION REPORTS ON GOOGLE DRIVE ===")
!ls -lh /content/drive/MyDrive/multicrop/reports/